# IngredientMassEstimation V5 — masa total auxiliar con MLP compartido

Experimento nuevo: **MLP compartido como V3 + cabezal auxiliar de masa total**.
Equivale a la ablación aux_only ya prevista, ahora con identidad V5 y resultados independientes.
No combina los MLP separados de V4 split_only. No promete mejora antes de medirla.

Mismos RGB, split provisional, vocabulario, cache de profundidad y presupuesto.
Todo entrenamiento/evaluación/test está desactivado al abrir. V3/V4 quedan en lectura.


## 1. Montar Drive

Usar la cuenta con acceso. Si está compartida, mantener el acceso directo en Mi unidad o adaptar DRIVE en punto 3.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shutil, zipfile, hashlib
from IPython.display import display, HTML
from google.colab import drive
drive.mount("/content/drive")



## 2. Descargar la rama V5 publicada

Publicar v5-auxiliary-mass antes de clonar, o adaptar BRANCH. Repositorio privado: PRIVATE_REPO=True y secreto GITHUB_TOKEN.


In [ ]:
URL = "https://github.com/marcogloazzo/IngredientMassEstimation.git"
BRANCH = "v5-auxiliary-mass"
PRIVATE_REPO = False
REPO = Path("/content/nutrition-code-v5")

def git_private(command):
    environment = os.environ.copy()
    try:
        if PRIVATE_REPO:
            from google.colab import userdata
            import base64
            token = userdata.get("GITHUB_TOKEN")
            authorization = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            environment.update({
                "GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraHeader",
                "GIT_CONFIG_VALUE_0": f"Authorization: Basic {authorization}",
            })
            del token, authorization
        result = subprocess.run(command, env=environment, capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError("Git falló: revisá URL, rama publicada y permisos del token. No se imprime la credencial.")
        return result.stdout.strip()
    finally:
        environment.clear()

if (REPO / ".git").is_dir():
    current = git_private(["git", "-C", str(REPO), "branch", "--show-current"])
    assert current == BRANCH, f"Esta carpeta tiene otra rama: {current}. Usá una carpeta nueva."
    git_private(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH])
else:
    git_private(["git", "clone", "--branch", BRANCH, "--single-branch", URL, str(REPO)])

PROJECT = next((p for p in (REPO, REPO / "IngredientMassEstimation")
                if (p / "config/query2label_100_v5.json").is_file()), None)
assert PROJECT is not None, "Falta publicar el código V5."
os.chdir(PROJECT)
print("Proyecto:", PROJECT)
print("Rama:", git_private(["git", "-C", str(REPO), "branch", "--show-current"]))

print("Commit:", git_private(["git", "-C", str(REPO), "rev-parse", "HEAD"]))



## 3. Dependencias y rutas V5

Configuración única, MLP compartido y auxiliar activo. No utilizar checkpoints V3/V4 como resume.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-v3.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DRIVE = Path("/content/drive/MyDrive/IngredientMassEstimation")
SOURCE_V3 = DRIVE / "v3"
SOURCE_V4 = DRIVE / "v4"
V5 = DRIVE / "v5"
EXPERIMENT_TAG = "run_01"
REPORT_TAG = "eval_01"
DIAGNOSTIC_TAG = "epoch39_01"
assert all("/" not in tag and "\\" not in tag and tag not in ("", ".", "..")
           for tag in (EXPERIMENT_TAG, REPORT_TAG, DIAGNOSTIC_TAG))
CONFIG = PROJECT / "config/query2label_100_v5.json"
cfg = json.loads(CONFIG.read_text())
assert cfg["split_task_mlps"] is False and cfg["auxiliary_total_mass"] is True
RUN = V5 / "runs" / f"query2label_100_v5_aux_{EXPERIMENT_TAG}"
VALIDATION = V5 / "reports" / f"{EXPERIMENT_TAG}_{REPORT_TAG}" / "validation"
TEST = VALIDATION.parent / "test"
CHECKPOINT = VALIDATION / "calibrated.pt"
V3_VALIDATION = SOURCE_V3 / "reports/run_01/validation"
V4_VALIDATION = SOURCE_V4 / "reports/split_only_run_01_eval_01/validation"
GATE_DIAGNOSTIC = V5 / "diagnostics" / f"v3_gate_{DIAGNOSTIC_TAG}"
LABELS_PARENT = Path("/content/v5_prepared")
PROCESSED = LABELS_PARENT / "processed_100_grouped_provisional"
DATA_PARENT = Path("/content/nutrition5k_v3")
DEPTH_CACHE = Path("/content/depth_v3_320")
DEPTH_ARCHIVE = SOURCE_V3 / "depth_v3_320.zip"

def run_stream(arguments):
    command = [str(arg) for arg in arguments]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
    finally:
        process.stdout.close()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

print("Device:", DEVICE, "| V5: MLP compartido + masa auxiliar")
print("RUN nuevo:", RUN)
print("Pesos nuevos:", {key: cfg["loss_weights"][key] for key in ("aux_mass", "mass_consistency")})
print("Fuentes históricas (solo lectura):", V3_VALIDATION, V4_VALIDATION)


## 4. Restaurar etiquetas agrupadas originales

Reutilizar el ZIP V3 y verificar sus hashes. No regenerar split ni vocabulario.


In [ ]:
from pathlib import PurePosixPath
import stat

def extract_zip_safe(archive, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as source:
        checked = []
        for info in source.infolist():
            name = info.filename.replace("\\", "/")
            relative = PurePosixPath(name)
            if relative.is_absolute() or not relative.parts or ":" in relative.parts[0]:
                raise ValueError(f"Nombre inseguro en ZIP: {name}")
            target = (destination / relative).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f"Ruta insegura en ZIP: {name}")
            if stat.S_ISLNK(info.external_attr >> 16):
                raise ValueError(f"No se aceptan enlaces en ZIP: {name}")
            checked.append((info, target, name.endswith("/")))
        for info, target, is_directory in checked:
            if is_directory:
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with source.open(info) as incoming, target.open("wb") as outgoing:
                    shutil.copyfileobj(incoming, outgoing)


labels_zip = SOURCE_V3 / "data/processed_100_grouped_provisional.zip"
assert labels_zip.is_file(), f"Falta el paquete existente de V3: {labels_zip}"
if not (PROCESSED / "records.jsonl").is_file():
    extract_zip_safe(labels_zip, LABELS_PARENT)
package = json.loads((LABELS_PARENT / "package_manifest.json").read_text())
for name, digest in package["file_sha256"].items():
    path = PROCESSED / name
    assert path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == digest, f"Etiquetas modificadas: {name}"
print("Etiquetas V3 conservadas:", PROCESSED)



## 5. Diagnóstico de referencia V3 — opcional, sin modelo

RUN_GATE_DIAGNOSTIC=True solo para reproducir el diagnóstico. Ya fue analizado; V5 no exige repetirlo antes de entrenar. Se escribe en v5/diagnostics/. Usar otro DIAGNOSTIC_TAG para repetir.


In [ ]:
RUN_GATE_DIAGNOSTIC = False
if RUN_GATE_DIAGNOSTIC:
    assert (V3_VALIDATION / "predictions.npz").is_file(), "Falta predictions.npz de validación V3."
    run_stream([sys.executable, "-u", "scripts/diagnose_mass_gate.py",
                "--predictions", V3_VALIDATION / "predictions.npz",
                "--details", V3_VALIDATION / "evaluation_details.json",
                "--vocabulary", PROCESSED / "vocabulary.json",
                "--output-dir", GATE_DIAGNOSTIC])
if (GATE_DIAGNOSTIC / "diagnostic.html").is_file():
    display(HTML((GATE_DIAGNOSTIC / "diagnostic.html").read_text(encoding="utf-8")))
else:
    print("Diagnóstico desactivado: habilitar RUN_GATE_DIAGNOSTIC.")



## 6. Restaurar RGB y reutilizar depth V3

Misma geometría 320×320, sin regenerar depth. RUN_DATA_CHECK=True verifica datos/cache sin ejecutar modelo ni entrenar.


In [ ]:
def dataset_roots():
    if not DATA_PARENT.is_dir():
        return []
    return sorted({folder.parent for folder in DATA_PARENT.rglob("metadata")
                   if folder.is_dir()
                   and (folder / "dish_metadata_cafe1.csv").is_file()
                   and (folder / "ingredients_metadata.csv").is_file()
                   and (folder.parent / "dish_ids").is_dir()
                   and (folder.parent / "imagery/realsense_overhead").is_dir()})

roots = dataset_roots()
if not roots:
    source = DRIVE / "nutrition5k_rgb.zip"
    assert source.is_file(), f"Falta: {source}"
    local = Path("/content/nutrition5k_rgb_v5.zip")
    shutil.copy2(source, local)
    extract_zip_safe(local, DATA_PARENT)
    roots = dataset_roots()
assert len(roots) == 1, f"Raíz RGB no única: {roots}"
DATA = roots[0]
if not (DEPTH_CACHE / "manifest.json").is_file():
    assert DEPTH_ARCHIVE.is_file(), "Falta el ZIP de mapas V3 320x320 en Drive."
    local = Path("/content/depth_v3_320.zip")
    shutil.copy2(DEPTH_ARCHIVE, local)
    extract_zip_safe(local, DEPTH_CACHE.parent)
assert (DEPTH_CACHE / "manifest.json").is_file(), "Revisá la estructura del ZIP de depth V3."
RUN_DATA_CHECK = False
if RUN_DATA_CHECK:
    run_stream([sys.executable, "-u", "scripts/train_image_model.py", "--config", CONFIG,
                "--dataset-root", DATA, "--processed-dir", PROCESSED, "--depth-cache", DEPTH_CACHE,
                "--device", DEVICE, "--check-data"])
print("RGB:", DATA, "| cache compatible:", DEPTH_CACHE)



## 7. Entrenamiento V5 — desactivado

Solo habilitar cuando decidas entrenar. MODE="new" para la primera corrida.
Máximo 100 épocas y paciencia 8 por L1 de ingredientes, igual que V3/V4.
Auxiliar: supervisión de masa (0,25) y consistencia con suma de ingredientes (0,10), normalizadas por media train.
Gramos/nutrientes finales siguen usando p × c. Resume solo desde last.pt de esta V5.


In [ ]:
RUN_TRAINING = False
MODE = "new"
TOTAL_EPOCHS = 100
if RUN_TRAINING:
    assert MODE in ("new", "resume")
    args = [sys.executable, "-u", "scripts/train_image_model.py", "--config", CONFIG,
            "--dataset-root", DATA, "--processed-dir", PROCESSED, "--depth-cache", DEPTH_CACHE,
            "--output-dir", RUN, "--torch-cache", DRIVE / "torch_cache",
            "--device", DEVICE, "--workers", "2", "--epochs", TOTAL_EPOCHS]
    if MODE == "resume":
        assert (RUN / "last.pt").is_file(), "Falta last.pt de esta V5."
        args += ["--resume", RUN / "last.pt"]
    else:
        assert not RUN.exists() or not any(RUN.iterdir()), "RUN tiene resultados: usar resume de V5 o un EXPERIMENT_TAG nuevo."
    run_stream(args)
else:
    print("Entrenamiento V5 desactivado.")


## 8. Validación, calibración y test — desactivados

Primero validación. Mantener RUN_TEST=False durante selección. Calibrar presencia no cambia gramos. Revisar MAE final de masa y nutrientes, además de L1 y métricas auxiliares.


In [ ]:
RUN_CALIBRATION = False
RUN_TEST = False
shared = ["--dataset-root", DATA, "--processed-dir", PROCESSED, "--depth-cache", DEPTH_CACHE,
          "--device", DEVICE, "--batch-size", "2", "--workers", "2"]
if RUN_CALIBRATION:
    assert (RUN / "best.pt").is_file(), "Falta best.pt de esta V5."
    run_stream([sys.executable, "-u", "scripts/evaluate_image_model.py", "--checkpoint", RUN / "best.pt",
                "--split", "validation", "--calibrate", "--output-dir", VALIDATION, *shared])
if RUN_TEST:
    assert CHECKPOINT.is_file(), "Primero calibrar con validación."
    run_stream([sys.executable, "-u", "scripts/evaluate_image_model.py", "--checkpoint", CHECKPOINT,
                "--split", "test", "--output-dir", TEST, *shared])
for split, folder in (("VALIDACIÓN", VALIDATION), ("TEST · solo referencia", TEST)):
    if (folder / "metrics.json").is_file():
        print(split)
        display(json.loads((folder / "metrics.json").read_text()))



## 9. Diagnóstico V5 y comparación con V3/V4 — archivos guardados

Mismos fingerprints y validación. Una mejora del cabezal auxiliar solo no demuestra mejora final; controlar composición, gramos falsos y nutrientes.


In [ ]:
RUN_V5_DIAGNOSTIC = False
RUN_COMPARISON = False
V5_DIAGNOSTIC = V5 / "diagnostics" / f"{EXPERIMENT_TAG}_{REPORT_TAG}"
if RUN_V5_DIAGNOSTIC:
    run_stream([sys.executable, "-u", "scripts/diagnose_mass_gate.py",
                "--predictions", VALIDATION / "predictions.npz",
                "--details", VALIDATION / "evaluation_details.json",
                "--vocabulary", PROCESSED / "vocabulary.json", "--output-dir", V5_DIAGNOSTIC])
if (V5_DIAGNOSTIC / "diagnostic.html").is_file():
    display(HTML((V5_DIAGNOSTIC / "diagnostic.html").read_text(encoding="utf-8")))
for label, previous in (("v3", V3_VALIDATION), ("v4_split_only", V4_VALIDATION)):
    comparison_dir = V5 / "comparisons" / f"{label}_vs_v5_{EXPERIMENT_TAG}_{REPORT_TAG}"
    if RUN_COMPARISON:
        assert (previous / "metrics.json").is_file(), f"Falta referencia: {previous}"
        run_stream([sys.executable, "-u", "scripts/compare_image_experiments.py",
                    "--left", previous, "--right", VALIDATION, "--output-dir", comparison_dir])
    if (comparison_dir / "comparison.html").is_file():
        print(label, "vs V5")
        display(HTML((comparison_dir / "comparison.html").read_text(encoding="utf-8")))


## Git y Drive

Git: rama v5-auxiliary-mass, código compatible actualizado, config/query2label_100_v5.json, tests/test_v5.py, documentos V5, README y este notebook limpio.
No incluir notebooks V3/V4 ejecutados ni datos/checkpoints/reportes. No hubo commit/push automático.

Drive: abrir/guardar este notebook V5. Reutilizar nutrition5k_rgb.zip, v3/data/processed_100_grouped_provisional.zip y v3/depth_v3_320.zip.
Conservar v3/ y v4/ completas. Resultados nuevos en v5/. No subir nuevamente pesos ni generar mapas.

V5 parte de una corrida nueva desde ImageNet como los controles, no resume de V3/V4. No cambia a ASL ni agrega módulos de atención/encoder/depth.
